# Oblig 2

- All exercises should be solved by splitting the dataset into 60% training, 20% validation,
and 20% test, using a stratified split so that the class ratio is preserved across
all three sets.

- Use the training set for 5-fold stratified cross-validation, the validation
set for model selection and hyperparameter tuning, and reserve the test set strictly
for final evaluation.

- For each experiment, report the mean and standard deviation
of accuracy (and, where specified, other metrics such as precision, recall, F1, or AUC)
among the folds. Given the class imbalance, accuracy alone is not sufficient; wherever it
is asked for, also report at least one imbalance-aware metric (e.g., balanced accuracy, F1,
or ROC-AUC).

In [1]:
import numpy as np
# import the dataset, and download for further use.
# Run this once to get the data

from ucimlrepo import fetch_ucirepo

def import_data() -> None:
    """
    Gets the dataset, and makes local csv
    """
    # fetch dataset
    adult = fetch_ucirepo(id=2)

    # data (as pandas dataframes)
    X = adult.data.features
    y = adult.data.targets

    X.to_csv("adult_features.csv", index=False)
    y.to_csv("adult_targets.csv", index=False)


In [2]:
# we now have the data locally as csv files
import pandas as pd

# import_data() # gets the dataset,

X = pd.read_csv("adult_features.csv") # features
y = pd.read_csv("adult_targets.csv")  # targets

print(X.shape)
print(X.dtypes)

(48842, 14)
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
dtype: object


In [3]:
from sklearn.model_selection import train_test_split

def train_val_test_split(X, y):
    """
    Create a 60, 20, 20 split for train, test and validation
    :param X: The set containing the features
    :param y: The set containing the targets
    :return: X_train, X_val, X_test, y_train, y_val, y_test
    """
    # First split into training+validation (80%) and test (20%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y  # Use stratify for classification
    )

    # Calculate the validation size relative to the train_val set
    # If test is 20%, validation is 20%, then validation is 20 / (100-20) = 20/80 = ~0.25
    val_size_relative = 0.20 / (1.0 - 0.20)

    # Split train_val into final training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_size_relative,
        random_state=42,
        stratify=y_train_val  # Use stratify again
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# https://apxml.com/courses/introduction-to-neural-networks/chapter-2-data-preparation-neural-networks/data-splitting

In [4]:
# The dataset without missing values, split into 60/20/20 splits
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y) # todo: replace with the non-null data

# check sizes
print(f"Original dataset size: {len(X)}")
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

#We now have a training set (60% of total data) which we can use StratifiedKFold on.

Original dataset size: 48842
Training set size: 29304
Validation set size: 9769
Test set size: 9769


## Data preparation

In [5]:
X_train.info()
# from .shape() we expect 48842 non-null values.
# we can see that there are missing values in the dataframe

<class 'pandas.DataFrame'>
Index: 29304 entries, 48092 to 30449
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             29304 non-null  int64
 1   workclass       28746 non-null  str  
 2   fnlwgt          29304 non-null  int64
 3   education       29304 non-null  str  
 4   education-num   29304 non-null  int64
 5   marital-status  29304 non-null  str  
 6   occupation      28745 non-null  str  
 7   relationship    29304 non-null  str  
 8   race            29304 non-null  str  
 9   sex             29304 non-null  str  
 10  capital-gain    29304 non-null  int64
 11  capital-loss    29304 non-null  int64
 12  hours-per-week  29304 non-null  int64
 13  native-country  29149 non-null  str  
dtypes: int64(6), str(8)
memory usage: 3.4 MB


In [6]:
X_train.head(5)
# we can see that missing data is encoded as ?

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
48092,22,Private,535027,Some-college,10,Never-married,Transport-moving,Unmarried,Black,Male,0,0,15,United-States
31972,43,Self-emp-inc,62026,Prof-school,15,Married-civ-spouse,Exec-managerial,Husband,White,Male,99999,0,40,United-States
36415,44,Private,227466,Some-college,10,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40,United-States
36832,32,State-gov,19513,Masters,14,Never-married,Prof-specialty,Not-in-family,Asian-Pac-Islander,Female,0,0,40,Japan
752,58,Self-emp-not-inc,87510,10th,6,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States


In [7]:
# Q1.1

def missing_features(X)-> None:
    """
    Identify which features have missing data, and how many

    The spesific dataset used here encodes missing data as "?"
    Chec
    :param X: Feature set
    """

    X_copy = X # work on copy of the set

    X_copy[X_copy == '?'] = np.nan # replace ? with Nan, for string columns this is missing data
    print(X_copy.isnull().sum()) # count the amount of missing values

missing_features(X_train)

age                  0
workclass         1633
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1639
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     505
dtype: int64


In [8]:
# Check correlation between missing occupation and workclass
def check_correlation(X, feature_1, feature_2, feature_3) -> None:
    """
    Given that both features have missing values. Checks the correlation between two features in a given dataset.
    :param X: dataset containing feature_1 and feature_2
    :param feature_1: first feature
    :param feature_2: second feature
    :return: None
    """
    X_relevant = X[[feature_1, feature_2, feature_3]]

    X_missing_corr= X_relevant.isnull().corr() # check correlation between missing values

    print(X_missing_corr)
    # the output will be dataframe (2x2)
    # Which tells if the two features are missing together
    # Value close to 1 means that the often are missing together
    # Value close to -1 means that they are mutually exclusive, ie feature_1 is missing and feature_2 exist
    # Close to 0 means no correlation, you can not tell anything about feature_2 given feature_1

check_correlation(X_train, "occupation", "workclass", "native-country")

                occupation  workclass  native-country
occupation        1.000000   0.998060        0.000861
workclass         0.998060   1.000000        0.000981
native-country    0.000861   0.000981        1.000000


From this we can see that the missing values are highly a missing occupation is highly associated with a missing workclass ($0.998 \approx 1 $). We can also see that missing native-country is unrealated to the other missing features.

In [9]:
missing_workclass = X_train['workclass'].isnull()
print(X_train.loc[missing_workclass, 'hours-per-week'].describe())

zero_hours_missing = (X_train.loc[X['workclass'].isnull(), 'hours-per-week'] == 0).sum()
print(f"Missing workclass AND 0 hours: {zero_hours_missing}")

count    1633.000000
mean       31.559094
std        14.974499
min         1.000000
25%        20.000000
50%        35.000000
75%        40.000000
max        99.000000
Name: hours-per-week, dtype: float64
Missing workclass AND 0 hours: 0


In [10]:
missing_workclass = X_train['workclass'].isnull()
print(X_train.loc[~missing_workclass, 'hours-per-week'].describe())

count    27671.000000
mean        40.879260
std         12.095009
min          1.000000
25%         40.000000
50%         40.000000
75%         45.000000
max         99.000000
Name: hours-per-week, dtype: float64


Based on [determine imputation](https://stats.stackexchange.com/questions/541337/how-should-i-determine-what-imputation-method-to-use) we conclude that the missing values are MAR (missing at random). Which means that "the probability the data are missing has something to do with variables". We have seen that the missing occupation correlates with missing workclass. There are also no row with missing workclass that is associated with 0 hours worked, which means that the correspondent is working. This suggests the missingness is better explained by other observed variables

We should not delete the rows with missing data as this would remove a non-random subset, it linked with occupation and correlated with a lower hrs/week. We decide on creating a missing category, this is because we are working with a categorical features, so mean imputation would not work. And because it is non-random it carries some information.

In [11]:
def replace_missing(X, feature_1, feature_2, feature_3) -> pd.DataFrame:
    """
    Replace missing data in rows with the column features_1-3, with a missing category
    :param X: Dataset
    :param feature_1: feature 1
    :param feature_2: feature 2
    :param feature_3: feature 3
    :return: New dataset with missing category
    """
    X['workclass'] = X[feature_1].fillna('Missing')
    X['occupation'] = X[feature_2].fillna('Missing')
    X['native-country'] = X[feature_3].fillna('Missing')

    return X

X_nonmissing = replace_missing(X_train, "occupation", "workclass", "native-country")

In [12]:
# Q1.2
def outliers_IQR(X, feature_list, verbose=True):
    """
    Detect outliers in numerical features using the IQR method.

    :param X: DataFrame containing the features
    :param feature_list: list of numerical column names to check
    :return: dict mapping feature name -> boolean mask (True = outlier)
    """
    outlier_masks = {}

    for col in feature_list:
        Q1 = X[col].quantile(0.25) # define Q1
        Q3 = X[col].quantile(0.75) # define Q3
        IQR = Q3 - Q1

        # create the bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        mask = (X[col] < lower_bound) | (X[col] > upper_bound) # create a mask to check if a row is above or below the bound
        outlier_masks[col] = mask # apply the mask

        if verbose:
            print(f"{col}: outliers={mask.sum()} ({mask.mean()*100:.2f}%)")

    return outlier_masks

int_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
outlier_masks = outliers_IQR(X_train, int_features)


age: outliers=135 (0.46%)
fnlwgt: outliers=862 (2.94%)
education-num: outliers=1082 (3.69%)
capital-gain: outliers=2414 (8.24%)
capital-loss: outliers=1361 (4.64%)
hours-per-week: outliers=8152 (27.82%)


In [13]:
def outliers_zscore(X, feature_list, threshold=3, verbose=True):
    """
    Detect outliers in numerical features using the Z-score method.

    :param X: DataFrame containing the features
    :param feature_list: list of numerical column names to check
    :param threshold: absolute z-score above which a point is considered an outlier
    :return: dict mapping feature name -> boolean mask (True = outlier)
    """
    outlier_masks = {}

    for col in feature_list:
        mean = X[col].mean()   # define mean
        std = X[col].std()     # define standard deviation

        # compute the z-scores
        z_scores = (X[col] - mean) / std

        mask = z_scores.abs() > threshold  # create a mask to check if a row exceeds the threshold
        outlier_masks[col] = mask          # apply the mask

        if verbose:
            print(f"{col}: outliers={mask.sum()} ({mask.mean()*100:.2f}%)")

    return outlier_masks

outlier_masks = outliers_zscore(X_train, int_features)

age: outliers=116 (0.40%)
fnlwgt: outliers=314 (1.07%)
education-num: outliers=211 (0.72%)
capital-gain: outliers=202 (0.69%)
capital-loss: outliers=1320 (4.50%)
hours-per-week: outliers=372 (1.27%)


Tree-based models are robust to outliers as they split on threshold. How much above the threshold does not matter. Keeping the outliers works fine for tree based models.

SVMs work on distances between points, the kernels are senstive to scale and spread of features. Thus we should cap the features for use with SVMs.

We should cap since the SVMs work poorly with outliers. We dont remove since we lose sample size, which both models respond poorly too.